# Find Patents

In [1]:
import pandas as pd
df = pd.read_csv("D:\\New-patents\\Landscaping\\data\\seed_and_antiseed_dataset.csv")
df

C:\Users\Asus\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,Num_patent,Similarity,flag_patent,grp_ml,grp_evo,grp_nlp,grp_speech,grp_vision,grp_kr,grp_planning,grp_hardware
0,7664556,0.090971,1,na,antiseed,na,na,na,na,l1,na
1,6199040,0.089747,1,na,na,l2,l1,na,na,antiseed,na
2,7885968,0.087807,1,na,na,l2,l2,na,l2,na,antiseed
3,5822496,0.087701,1,na,na,l2,na,l1,l1,l1,seed
4,6134344,0.087036,1,l1,l2,l2,l2,seed,l1,l2,l2
...,...,...,...,...,...,...,...,...,...,...,...
72430,5139541,-0.029689,1,na,antiseed,na,na,na,na,na,na
72431,4366272,-0.030128,1,na,na,na,antiseed,na,na,na,na
72432,8552104,-0.030188,1,na,na,na,na,na,na,na,antiseed
72433,4428840,-0.030381,1,na,antiseed,na,na,na,na,na,na


In [2]:
def determine_is_ai(row):
    if 'seed' in row.values:
        return 1
    elif 'antiseed' in row.values:
        return 0
    return pd.NA

# Applying the function row-wise
df['is_ai'] = df.apply(determine_is_ai, axis=1)

In [9]:
len(df[df['grp_evo'] == 'seed'])

32

In [10]:
df

,Num_patent,Similarity,flag_patent,grp_ml,grp_evo,grp_nlp,grp_speech,grp_vision,grp_kr,grp_planning,grp_hardware,is_ai
0,7664556,0.090971,1,na,antiseed,na,na,na,na,l1,na,0
1,6199040,0.089747,1,na,na,l2,l1,na,na,antiseed,na,0
2,7885968,0.087807,1,na,na,l2,l2,na,l2,na,antiseed,0
3,5822496,0.087701,1,na,na,l2,na,l1,l1,l1,seed,1
4,6134344,0.087036,1,l1,l2,l2,l2,seed,l1,l2,l2,1
...,...,...,...,...,...,...,...,...,...,...,...,...
72430,5139541,-0.029689,1,na,antiseed,na,na,na,na,na,na,0
72431,4366272,-0.030128,1,na,na,na,antiseed,na,na,na,na,0
72432,8552104,-0.030188,1,na,na,na,na,na,na,na,antiseed,0
72433,4428840,-0.030381,1,na,antiseed,na,na,na,na,na,na,0


In [ ]:
num_patents_seed = []
num_patents_antiseed = []

data_seed = df[df['is_ai'] == 1].sample(400)['Num_patent']
data_antiseed = df[df['is_ai'] == 0].sample(600)['Num_patent']
for i in range(len(data_seed)):
    num_patents_seed.append(data_seed.iloc[i])
for i in range(len(data_antiseed)):
    num_patents_antiseed.append(data_antiseed.iloc[i])

In [7]:
num_patents = num_patents_seed + num_patents_antiseed

# load patent texts

In [ ]:
g =open("D:\\New-patents\\Landscaping\\LDA phrases\\data\\fulltext_concat.tsv")
l=g.readline()# header
patent_texts = {}
i=0
while (True):
    l=g.readline()
    s=l.split()
    if len(s) > 10:
        q =2
        n = s[1]
        if (n.isnumeric() == False) or (len(n) < 3):
            continue
        for id in num_patents:
            if id == int(n):
                patent_texts[id] = " ".join(s[q:])
                print(len(patent_texts))
        if len(patent_texts) == 100:
            break
    

In [ ]:
sorted_d = {}
for k in num_patents:
    sorted_d[int(k)] = patent_texts[k]


{8909570: "The following patent applications are also incorporated herein for their teachings: U.S. patent application Ser. No. 12/267,287, filed 7 Nov. 2008, entitled “DISTRIBUTED NETWORK FOR PERFORMING COMPLEX ALGORITHMS”; U.S. patent application Ser. No. 12/769,589, filed: 28 Apr. 2010, entitled “DISTRIBUTED EVOLUTIONARY ALGORITHM FOR ASSET MANAGEMENT AND TRADING”; and U.S. patent application Ser. No. 12/769,605, filed 28 Apr. 2010, entitled “CLASS-BASED DISTRIBUTED EVOLUTIONARY ALGORITHM FOR ASSET MANAGEMENT AND TRADING”. BACKGROUND The invention relates generally to data mining, and more particularly, to the use of genetic algorithms to extract useful rules or relationships from a data set for use in controlling systems. In many environments, a large amount of data can be or has been collected which records experience over time within the environment. For example, a healthcare environment may record clinical data, diagnoses and treatment regimens for a large number of patients, as

In [81]:
import pickle
f=open('./data/100_samples_seed', 'wb')
pickle.dump(sorted_d,f)
f.close()

# LLM model

In [94]:
import json
import re
from llama_cpp import Llama

def load_llm(model_path: str, n_ctx: int = 16384, n_gpu_layers: int = -1):
    return Llama(model_path=model_path, n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, verbose=False)

def extract_json_robust(text: str) -> dict:
    """Multi-strategy JSON extraction for LLM outputs"""
    # 1. Strip markdown
    cleaned = re.sub(r'^```(?:json)?\s*', '', text)
    cleaned = re.sub(r'\s*```$', '', cleaned).strip()
    
    # 2. Fix Python literals
    cleaned = re.sub(r'\bTrue\b', 'true', cleaned)
    cleaned = re.sub(r'\bFalse\b', 'false', cleaned)
    cleaned = re.sub(r'\bNone\b', 'null', cleaned)
    
    # 3. Close unclosed brackets/braces
    stack = []
    for char in cleaned:
        if char in '{[': stack.append(char)
        elif char == '}': 
            if stack and stack[-1] == '{': stack.pop()
        elif char == ']': 
            if stack and stack[-1] == '[': stack.pop()
    cleaned += '}' * stack.count('{') + ']' * stack.count('[')
    
    # 4. Remove trailing commas
    cleaned = re.sub(r',\s*([\]}])', r'\1', cleaned)
    
    # 5. Try direct parse
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    
    # 6. Fallback: extract known keys via regex
    keys = ["ai_score", "ai_role", "score_rationale", "verification_note"]
    fallback = {}
    for k in keys:
        match = re.search(rf'"{k}"\s*:\s*(".*?"|[\d.]+|true|false|null)', cleaned)
        if match:
            val = match.group(1).strip('"')
            fallback[k] = val if val not in ("true", "false", "null") else json.loads(val)
    
    # Extract phrases array
    phrases = re.findall(r'\{"text"\s*:\s*"([^"]*)"[^}]*\}', cleaned)
    if phrases:
        fallback["verbatim_ai_phrases"] = [{"text": p, "confidence": 0.5} for p in phrases]
    
    return fallback if fallback else None

def analyze_patent(llm: Llama, patent_text: str) -> dict:
    system_prompt = "You are a precise AI patent analyst. Output ONLY raw JSON. No markdown. No explanations."
    user_prompt = f"""You are an expert AI patent analyst. Analyze the provided patent text (Brief Description + Claims) and assess its relationship to Artificial Intelligence (AI).

OUTPUT ONLY RAW JSON. NO MARKDOWN. NO EXPLANATIONS.

{{
  "ai_score": <float 0.0-1.0>,
  "score_rationale": "<1-2 sentences explaining score using the rubric below>",
  "ai_subfields": ["<list of AI domains, e.g., 'computer vision', 'reinforcement learning', 'optimization'>"],
  "theoretical_aspects": ["<list of theoretical AI elements: novel architectures, mathematical formulations, training paradigms, loss functions, etc.>"],
  "applied_aspects": ["<list of applied AI elements: model integration, inference pipelines, data preprocessing, deployment, API usage, etc.>"],
  "ai_role": "<core_invention | enabling_technology | peripheral_tool | none>",
  "verbatim_ai_phrases": [
    {{
      "text": "<EXACT verbatim phrase or complete sentence from input>",
      "section": "<Claims | Brief_Description | Background | Summary>",
      "confidence": <0.0-1.0>
    }}
  ],
  "non_ai_exclusions": ["<list of non-AI technical elements that might be confused with AI, e.g., 'traditional PID control', 'rule-based routing', 'hardware sensor fusion'>"],
  "verification_note": "<1 sentence confirming all phrases exist verbatim in the provided text>"
}}

SCORING RUBRIC:
• 0.0–0.3: AI is incidental, mentioned once, or used as a generic tool with no technical detail.
• 0.4–0.6: AI is present and described, but peripheral to the invention's novelty (e.g., "AI optimizes X" where X is the core).
• 0.7–0.85: AI is central to the technical solution, with clear implementation details or architectural novelty.
• 0.9–1.0: Core invention is an AI method, model, or system; AI defines the novelty and claims.
Note: Weight Claims heavily. AI described only in examples or background should score lower.

STRICT RULES:
1. EXTRACT ONLY EXACT VERBATIM PHRASES OR COMPLETE SENTENCES. Do not extract isolated words, paraphrase, or summarize.
2. All JSON keys listed above MUST be present. If no AI is found, set ai_score=0.0, ai_role="none", and return empty arrays for lists.
3. Use precise AI terminology: neural networks, training, inference, loss, gradient, embedding, clustering, generative, reinforcement, optimization, pattern recognition, model deployment, API integration, etc.
4. Output MUST be valid JSON. Do not add explanations, markdown, or trailing text after the JSON object.
5. Keep output concise. Do not exceed 1500 tokens.

Patent Text:
{patent_text}
"""

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    
    output = llm.create_chat_completion(
        messages=messages,
        max_tokens=4096,
        temperature=0.0,
        top_p=0.9,
        repeat_penalty=1.1,
        response_format={"type": "json_object"}
    )
    
    raw = output["choices"][0]["message"]["content"]
    result = extract_json_robust(raw)
    
    # Validate structure
    required = ["ai_score", "ai_role", "verbatim_ai_phrases", "score_rationale"]
    missing = [k for k in required if k not in (result or {})]
    
    if not result or missing:
        print("\n=== RAW LLM OUTPUT ===")
        print(raw)
        print("\n=== DIAGNOSTICS ===")
        print(f"Length: {len(raw)} chars")
        print(f"Missing keys: {missing}")
        print(f"Starts with: {raw[:50]}")
        print(f"Ends with: ...{raw[-50:]}")
        print(f"Contains '```': {'```' in raw}")
        return {"error": "JSON structure incomplete or invalid", "missing_keys": missing, "raw_length": len(raw)}
    
    return result

In [ ]:
# # ================= USAGE =================
MODEL_PATH = "C:\\Users\\Asus\\.lmstudio\\models\\lmstudio-community\\Qwen3.6-35B-A3B-GGUF\\Qwen3.6-35B-A3B-Q4_K_M.gguf"  # Adjust path
PATENT_TEXT = f"""[{sorted_d[8775341][35000:70000]}]"""
llm = load_llm(MODEL_PATH, n_ctx=16384)
result = analyze_patent(llm, PATENT_TEXT)
print(json.dumps(result, indent=2))

llama_context: n_ctx_seq (16384) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


{
  "ai_score": 0.95,
  "score_rationale": "The patent's core invention is a 'cognitive hierarchical stacked neural network' designed to emulate human cognitive development for spam detection, making AI the central novelty and defining element of the claims.",
  "ai_subfields": [
    "neural networks",
    "machine learning",
    "pattern recognition",
    "reinforcement learning"
  ],
  "theoretical_aspects": [
    "Hierarchical stacked computer neural networks based on Commons et al.'s Model of Hierarchical Complexity",
    "Negative power function discounting for past events",
    "Training at each stage with valued consequences delivered at each layer",
    "Simulating cognitive development in the human brain"
  ],
  "applied_aspects": [
    "Detecting aberrant messages, such as email spam",
    "Spam/clean classification using Bayesian classifiers or support vector machines",
    "Integration of neural networks into robotic droids and manufacturing systems",
    "Real-time pattern

# create dataset

In [ ]:
import pickle
import pandas as pd
f=open("./data/100_samples_seed", "rb")
sorted_d=pickle.load(f)
f.close()

In [99]:
sorted_d

{8909570: "The following patent applications are also incorporated herein for their teachings: U.S. patent application Ser. No. 12/267,287, filed 7 Nov. 2008, entitled “DISTRIBUTED NETWORK FOR PERFORMING COMPLEX ALGORITHMS”; U.S. patent application Ser. No. 12/769,589, filed: 28 Apr. 2010, entitled “DISTRIBUTED EVOLUTIONARY ALGORITHM FOR ASSET MANAGEMENT AND TRADING”; and U.S. patent application Ser. No. 12/769,605, filed 28 Apr. 2010, entitled “CLASS-BASED DISTRIBUTED EVOLUTIONARY ALGORITHM FOR ASSET MANAGEMENT AND TRADING”. BACKGROUND The invention relates generally to data mining, and more particularly, to the use of genetic algorithms to extract useful rules or relationships from a data set for use in controlling systems. In many environments, a large amount of data can be or has been collected which records experience over time within the environment. For example, a healthcare environment may record clinical data, diagnoses and treatment regimens for a large number of patients, as

In [100]:
patent_id = []
patent_text = []
ai_score = []
theoretical_aspects = []
applied_aspects = []
ai_role = []
verbatim_ai_phrases = []
non_ai_exclusions = []
verification_note = []
dataset = pd.DataFrame()


In [ ]:
MODEL_PATH = "C:\\Users\\Asus\\.lmstudio\\models\\lmstudio-community\\Qwen3.6-35B-A3B-GGUF\\Qwen3.6-35B-A3B-Q4_K_M.gguf"  # Adjust path
i=0
for id in list(sorted_d.keys()):
    # i = i+1
    # if i <=3:
    #     continue
    
    print(id)
    text = sorted_d[id]
    if len(text) > 35000:
        ch = len(text)//35000
        for i in range(ch+1):
            if i ==ch:
                text1 = text[(i*35000):]
            else:
                text1 = text[(i*35000):((i+1)*35000)]
                
            PATENT_TEXT = f"""[{text1}]"""
            llm = load_llm(MODEL_PATH, n_ctx=16384)
            result = analyze_patent(llm, PATENT_TEXT)
            a = json.dumps(result, indent=2)
            result_dict = json.loads(a)
            
            patent_id.append(str(id)+f'_{i}')
            patent_text.append(text1)
            ai_score.append(result_dict['ai_score'])
            theoretical_aspects.append(result_dict['theoretical_aspects'])
            applied_aspects.append(result_dict['applied_aspects'])
            ai_role.append(result_dict['ai_role'])
            verbatim_ai_phrases.append(result_dict['verbatim_ai_phrases'])
            non_ai_exclusions.append(result_dict['non_ai_exclusions'])
            verification_note.append(result_dict['verification_note'])
    else:
        PATENT_TEXT = f"""[{text}]"""
        llm = load_llm(MODEL_PATH, n_ctx=16384)
        result = analyze_patent(llm, PATENT_TEXT)
        a = json.dumps(result, indent=2)
        result_dict = json.loads(a)

        patent_id.append(id)
        patent_text.append(text)
        ai_score.append(result_dict['ai_score'])
        theoretical_aspects.append(result_dict['theoretical_aspects'])
        applied_aspects.append(result_dict['applied_aspects'])
        ai_role.append(result_dict['ai_role'])
        verbatim_ai_phrases.append(result_dict['verbatim_ai_phrases'])
        non_ai_exclusions.append(result_dict['non_ai_exclusions'])
        verification_note.append(result_dict['verification_note'])

In [102]:
dataset["patent_id"] = patent_id
dataset["patent_text"] = patent_text
dataset["ai_score"] = ai_score
dataset["theoretical_aspects"] = theoretical_aspects
dataset["applied_aspects"] = applied_aspects
dataset["ai_role"] = ai_role
dataset["verbatim_ai_phrases"] = verbatim_ai_phrases
dataset["non_ai_exclusions"] = non_ai_exclusions
dataset["verification_note"] = verification_note
dataset

,patent_id,patent_text,ai_score,theoretical_aspects,applied_aspects,ai_role,verbatim_ai_phrases,non_ai_exclusions,verification_note
0,8909570,The following patent applications are also inc...,0.85,"[Age-Layered Population Structure (ALPS), elit...",[deployment of selected evolved rules for cont...,core_invention,[{'text': 'The invention relates generally to ...,"[traditional rule-based systems, hardware sens...",All verbatim phrases are extracted exactly as ...
1,8751416,CROSS-REFERENCE TO RELATED APPLICATIONS The pr...,0.75,[generation of a probabilistic model from a de...,[MLA operative to receive data from the VDM an...,core_invention,[{'text': 'a machine learning algorithm (MLA) ...,[dynamic Bayesian Network (DBN) derived from o...,All phrases are extracted verbatim from the pr...
2,8170963,CROSS REFERENCES TO RELATED APPLICATIONS The p...,0.75,[genetic algorithm with crossover and mutation...,[processing detection data using learned featu...,core_invention,[{'text': 'The genetic search method in the re...,[traditional linear combination coefficient es...,All verbatim phrases are extracted exactly fro...
3,8775341_0,FIELD OF THE INVENTION The present invention r...,0.95,[hierarchical stacked neural networks configur...,[training neural network models using gradient...,core_invention,[{'text': 'The present invention relates to th...,"[traditional PID control (not mentioned), rule...",All verbatim phrases are extracted exactly as ...
4,8775341_1,re applied across the entire space of possible...,0.95,[Hierarchical stacked computer neural networks...,"[Detecting aberrant messages, such as email sp...",core_invention,[{'text': 'The present invention relates to th...,"[Traditional PID control mechanisms, Rule-base...",All verbatim phrases are extracted exactly as ...
...,...,...,...,...,...,...,...,...,...
103,7789091,BACKGROUND OF THE INVENTION 1. Field of the In...,0.00,[],[],none,[],"[abrasive coating, flexible screen member, mec...",All phrases exist verbatim in the provided text
104,8435730,FIELD OF THE INVENTION The invention relates t...,0.00,[],[],none,[],"[chemical composition formulation, cryogenic p...",All phrases exist verbatim in the provided text
105,5062968,"""BACKGROUND OF THE INVENTION The apparatus and...",0.00,[],[],none,[],"[mechanical filtration, pressurized gas system...",All phrases exist verbatim in the provided text.
106,8151913,BACKGROUND OF THE INVENTION 1. Field of the In...,0.00,[],[],none,[],"[differential mechanism, gear reduction system...",All phrases exist verbatim in the provided text.


In [103]:
dataset.to_csv("./data/100_ai_seed_samples_data.csv")